In [0]:
from pyspark.sql import functions as F

CATALOGO = "v_credit_medallion"
SILVER = f"{CATALOGO}.silver"
GOLD = f"{CATALOGO}.gold"

print("Camada Gold preparada.")
print(f"Origem: {SILVER}")
print(f"Destino: {GOLD}")

Camada Gold preparada.
Origem: v_credit_medallion.silver
Destino: v_credit_medallion.gold


In [0]:


# Carregar as tabelas da camada Silver
chamados = spark.table(f"{SILVER}.chamados")
custos = spark.table(f"{SILVER}.custos")
motivos = spark.table(f"{SILVER}.base_motivos")

# Consolidar os dados sem excluir chamados que não possuem custo
gold_chamados = (
    chamados
    .join(
        motivos.select(
            "id_motivo",
            "nome_motivo",
            "categoria",
            "criticidade"
        ),
        on="id_motivo",
        how="left"
    )
    .join(
        custos.select("id_chamado", "custo"),
        on="id_chamado",
        how="left"
    )
    .withColumn(
        "tem_custo",
        F.when(F.col("custo").isNotNull(), 1).otherwise(0)
    )
)


id_chamado,id_motivo,id_cliente,canal,resolvido,tempo_espera,tempo_atendimento,id_atendente,nome_motivo,categoria,criticidade,custo,tem_custo
89333,15,52402176464,chatbot,0,0.0,104.0,3,Renegociação de dívida,financeiro,alta,0.60,1
107195,6,59595994444,chatbot,0,0.0,192.0,6,Desbloqueio de cartão,cartao,alta,0.91,1
77695,11,83694843708,ura,1,0.0,138.0,19,Bloqueio de cartão,cartao,alta,1.83,1
128920,11,45998304098,chatbot,1,0.0,52.0,19,Bloqueio de cartão,cartao,alta,0.46,1
54487,12,18962962744,atendimento inicial,1,221.0,354.0,10,Contratação de cartão adicional,cartao,media,10.14,1
37471,12,56677484936,ura,0,0.0,341.0,20,Contratação de cartão adicional,cartao,media,2.78,1
46569,5,36600471753,ura,1,0.0,307.0,3,Consulta de limite,financeiro,baixa,2.14,1
116648,5,74083785931,chatbot,1,0.0,112.0,22,Consulta de limite,financeiro,baixa,0.51,1
44869,15,56910618092,chatbot,0,0.0,238.0,3,Renegociação de dívida,financeiro,alta,0.94,1
129000,9,99239593080,atendimento inicial,0,728.0,null,11,Problema com aplicativo,atendimento,media,2.08,1


In [0]:

# Garantir que o schema Gold existe
spark.sql("CREATE SCHEMA IF NOT EXISTS v_credit_medallion.gold")

# Salvar a tabela analítica
(
    gold_chamados.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("v_credit_medallion.gold.analise_chamados")
)

print("Tabela Gold criada com sucesso!")


Tabela Gold criada com sucesso!


In [0]:
tabela_gold = spark.table(
    "v_credit_medallion.gold.analise_chamados"
)

print("Total de registros:", tabela_gold.count())

display(tabela_gold.limit(10))

Total de registros: 131838


id_chamado,id_motivo,id_cliente,canal,resolvido,tempo_espera,tempo_atendimento,id_atendente,nome_motivo,categoria,criticidade,custo,tem_custo
89333,15,52402176464,chatbot,0,0.0,104.0,3,Renegociação de dívida,financeiro,alta,0.60,1
107195,6,59595994444,chatbot,0,0.0,192.0,6,Desbloqueio de cartão,cartao,alta,0.91,1
77695,11,83694843708,ura,1,0.0,138.0,19,Bloqueio de cartão,cartao,alta,1.83,1
128920,11,45998304098,chatbot,1,0.0,52.0,19,Bloqueio de cartão,cartao,alta,0.46,1
54487,12,18962962744,atendimento inicial,1,221.0,354.0,10,Contratação de cartão adicional,cartao,media,10.14,1
37471,12,56677484936,ura,0,0.0,341.0,20,Contratação de cartão adicional,cartao,media,2.78,1
46569,5,36600471753,ura,1,0.0,307.0,3,Consulta de limite,financeiro,baixa,2.14,1
116648,5,74083785931,chatbot,1,0.0,112.0,22,Consulta de limite,financeiro,baixa,0.51,1
44869,15,56910618092,chatbot,0,0.0,238.0,3,Renegociação de dívida,financeiro,alta,0.94,1
129000,9,99239593080,atendimento inicial,0,728.0,null,11,Problema com aplicativo,atendimento,media,2.08,1


In [0]:

from pyspark.sql import functions as F

gold_indicadores = (
    tabela_gold
    .groupBy("categoria", "nome_motivo", "canal")
    .agg(
        F.countDistinct("id_chamado").alias("total_chamados"),

        F.sum(
            F.when(F.col("resolvido") == 1, 1).otherwise(0)
        ).alias("chamados_resolvidos"),

        F.avg(
            F.when(
                F.col("resolvido").isin(0, 1),
                F.col("resolvido")
            )
        ).alias("taxa_resolucao"),

        F.avg("tempo_espera").alias("tempo_medio_espera"),

        F.avg("tempo_atendimento").alias("tempo_medio_atendimento"),

        F.sum("custo").alias("custo_total_registrado"),

        F.avg("custo").alias("custo_medio_registrado"),

        F.sum(
            F.when(F.col("tem_custo") == 0, 1).otherwise(0)
        ).alias("chamados_sem_custo")
    )
    .withColumn(
        "taxa_resolucao_pct",
        F.round(F.col("taxa_resolucao") * 100, 2)
    )
    .withColumn(
        "tempo_medio_espera",
        F.round("tempo_medio_espera", 2)
    )
    .withColumn(
        "tempo_medio_atendimento",
        F.round("tempo_medio_atendimento", 2)
    )
    .withColumn(
        "custo_total_registrado",
        F.round("custo_total_registrado", 2)
    )
    .withColumn(
        "custo_medio_registrado",
        F.round("custo_medio_registrado", 2)
    )
)

display(
    gold_indicadores
    .orderBy(F.desc("total_chamados"))
)


categoria,nome_motivo,canal,total_chamados,chamados_resolvidos,taxa_resolucao,tempo_medio_espera,tempo_medio_atendimento,custo_total_registrado,custo_medio_registrado,chamados_sem_custo,taxa_resolucao_pct
financeiro,Consulta de fatura,ura,6118,3796,0.6320346320346321,0.0,211.09,11133.12,1.85,90,63.2
financeiro,Consulta de fatura,chatbot,5589,3738,0.678279804028307,0.0,111.43,3154.73,0.57,63,67.83
financeiro,Consulta de limite,ura,4985,3005,0.6122656886715566,0.0,210.1,9068.83,1.84,61,61.23
atendimento,Problema com aplicativo,atendimento inicial,4928,1898,0.39125953411667697,357.07,361.87,66590.39,13.67,55,39.13
cartao,Compra não autorizada,atendimento inicial,4750,1729,0.3684210526315789,348.9,362.56,64672.28,13.76,50,36.84
cartao,Bloqueio de cartão,ura,4637,2649,0.5801576872536137,0.0,208.92,8449.02,1.84,46,58.02
financeiro,Consulta de limite,chatbot,4589,2975,0.6589147286821705,0.0,115.17,2628.98,0.58,48,65.89
financeiro,Renegociação de dívida,atendimento inicial,4473,1392,0.31614808085396323,345.73,374.68,60112.41,13.63,64,31.61
cartao,Bloqueio de cartão,chatbot,4192,2657,0.6445900048520136,0.0,112.79,2378.41,0.57,46,64.46
financeiro,Consulta de fatura,atendimento inicial,4050,2900,0.7282772476142642,344.68,358.98,54921.71,13.73,50,72.83


In [0]:

#Persistência dos indicadores de atendimento

(
    gold_indicadores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.indicadores_atendimento")
)

print("Tabela de indicadores salva com sucesso.")


Tabela de indicadores salva com sucesso.


In [0]:

# Análise de satisfação por categoria, motivo e canal

pesquisas = spark.table(f"{SILVER}.pesquisa_satisfacao")

# Consolidar as pesquisas por chamado para evitar duplicações
pesquisa_por_chamado = (
    pesquisas
    .groupBy("id_chamado")
    .agg(
        F.avg("nota_atendimento").alias("nota_media"),
        F.count("id_pesquisa").alias("quantidade_pesquisas")
    )
)

# Relacionar satisfação aos dados consolidados da Gold
gold_satisfacao = (
    tabela_gold
    .join(pesquisa_por_chamado, on="id_chamado", how="left")
    .groupBy("categoria", "nome_motivo", "canal")
    .agg(
        F.countDistinct("id_chamado").alias("total_chamados"),
        F.count("nota_media").alias("chamados_com_avaliacao"),
        F.round(F.avg("nota_media"), 2).alias("nota_media_satisfacao"),
        F.round(
            F.avg(
                F.when(F.col("nota_media") >= 4, 1.0)
                 .when(F.col("nota_media").isNotNull(), 0.0)
            ) * 100,
            2
        ).alias("avaliacoes_positivas_pct")
    )
)

# Salvar a tabela Gold
(
    gold_satisfacao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.indicadores_satisfacao")
)

display(
    spark.table(f"{GOLD}.indicadores_satisfacao")
    .orderBy(F.desc("total_chamados"))
)


categoria,nome_motivo,canal,total_chamados,chamados_com_avaliacao,nota_media_satisfacao,avaliacoes_positivas_pct
financeiro,Consulta de fatura,ura,6118,3448,4.48,88.72
financeiro,Consulta de fatura,chatbot,5589,3138,4.55,90.5
financeiro,Consulta de limite,ura,4985,2824,4.45,88.42
atendimento,Problema com aplicativo,atendimento inicial,4928,2684,3.49,52.12
cartao,Compra não autorizada,atendimento inicial,4750,2631,3.56,55.0
cartao,Bloqueio de cartão,ura,4637,2702,4.45,88.45
financeiro,Consulta de limite,chatbot,4589,2590,4.54,90.42
financeiro,Renegociação de dívida,atendimento inicial,4473,2416,3.47,51.49
cartao,Bloqueio de cartão,chatbot,4192,2352,4.53,90.65
financeiro,Consulta de fatura,atendimento inicial,4050,2239,3.87,67.75


In [0]:

# Painel executivo consolidado

indicadores_atendimento = spark.table(f"{GOLD}.indicadores_atendimento")
indicadores_satisfacao = spark.table(f"{GOLD}.indicadores_satisfacao")

# Remover a contagem duplicada para manter uma única coluna de volume
satisfacao_selecionada = indicadores_satisfacao.select(
    "categoria",
    "nome_motivo",
    "canal",
    "chamados_com_avaliacao",
    "nota_media_satisfacao",
    "avaliacoes_positivas_pct"
)

painel_executivo = indicadores_atendimento.join(
    satisfacao_selecionada,
    on=["categoria", "nome_motivo", "canal"],
    how="left"
)

(
    painel_executivo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.painel_executivo")
)

print("Painel executivo salvo na camada Gold.")
display(
    spark.table(f"{GOLD}.painel_executivo")
    .orderBy(F.desc("total_chamados"))
)


Painel executivo salvo na camada Gold.


categoria,nome_motivo,canal,total_chamados,chamados_resolvidos,taxa_resolucao,tempo_medio_espera,tempo_medio_atendimento,custo_total_registrado,custo_medio_registrado,chamados_sem_custo,taxa_resolucao_pct,chamados_com_avaliacao,nota_media_satisfacao,avaliacoes_positivas_pct
financeiro,Consulta de fatura,ura,6118,3796,0.6320346320346321,0.0,211.09,11133.12,1.85,90,63.2,3448,4.48,88.72
financeiro,Consulta de fatura,chatbot,5589,3738,0.678279804028307,0.0,111.43,3154.73,0.57,63,67.83,3138,4.55,90.5
financeiro,Consulta de limite,ura,4985,3005,0.6122656886715566,0.0,210.1,9068.83,1.84,61,61.23,2824,4.45,88.42
atendimento,Problema com aplicativo,atendimento inicial,4928,1898,0.39125953411667697,357.07,361.87,66590.39,13.67,55,39.13,2684,3.49,52.12
cartao,Compra não autorizada,atendimento inicial,4750,1729,0.3684210526315789,348.9,362.56,64672.28,13.76,50,36.84,2631,3.56,55.0
cartao,Bloqueio de cartão,ura,4637,2649,0.5801576872536137,0.0,208.92,8449.02,1.84,46,58.02,2702,4.45,88.45
financeiro,Consulta de limite,chatbot,4589,2975,0.6589147286821705,0.0,115.17,2628.98,0.58,48,65.89,2590,4.54,90.42
financeiro,Renegociação de dívida,atendimento inicial,4473,1392,0.31614808085396323,345.73,374.68,60112.41,13.63,64,31.61,2416,3.47,51.49
cartao,Bloqueio de cartão,chatbot,4192,2657,0.6445900048520136,0.0,112.79,2378.41,0.57,46,64.46,2352,4.53,90.65
financeiro,Consulta de fatura,atendimento inicial,4050,2900,0.7282772476142642,344.68,358.98,54921.71,13.73,50,72.83,2239,3.87,67.75
